In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
import os

In [3]:
def train_intent_model():
    print("Membaca dataset chat_dataset.csv...")
    dataset_path = "data/chat_dataset.csv"
    
    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None
        
    df = pd.read_csv(dataset_path)
    
    df = df.dropna()
    
    X = df['teks_chat']
    y = df['label_intent']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    
    # Membuat pipeline: TF-IDF (ubah teks jadi angka) -> SVM (klasifikasi intent)
    print("Melatih model NLU (TF-IDF + SVM)...")
    model = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', probability=True))
    
    # Proses Training
    model.fit(X_train, y_train)
    
    # Evaluasi Akurasi Model
    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    # Simpan model agar tidak perlu dilatih ulang setiap kali server menyala
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier.pkl")
    print("Model berhasil disimpan di models/intent_classifier.pkl\n")
    
    return model

In [4]:
def predict_intent(chat_text):
    # Load model yang sudah pintar
    model_path = "models/intent_classifier.pkl"
    if not os.path.exists(model_path):
        model = train_intent_model()
    else:
        model = joblib.load(model_path)
    
    # Prediksi intent dari chat baru
    prediksi = model.predict([chat_text])[0]
    
    # Ambil nilai probabilitas/keyakinan model (dalam persentase)
    probabilitas = max(model.predict_proba([chat_text])[0]) * 100
    
    return prediksi, probabilitas

In [5]:
train_intent_model()
    
print("--- SIMULASI TESTING AI 1 DI DALAM GAME ---")
test_chats = [
    "Jagain rumah gw pak pol, gw bayar mahal nih pake koin",
    "Lu curigaan mulu sama gw anjir, gw cuma warga biasa",
    "Woy si Budi dari tadi diem aja, fix dia ketuanya"
]

for chat in test_chats:
    intent, prob = predict_intent(chat)
    print(f"Chat Player: '{chat}'")
    print(f" > AI 1 Menebak: [{intent.upper()}] (Tingkat Keyakinan: {prob:.2f}%)\n")

Membaca dataset chat_dataset.csv...
Total data latih: 620 | Total data uji: 155
Melatih model NLU (TF-IDF + SVM)...

--- Hasil Ujian Model (Evaluasi) ---
                      precision    recall  f1-score   support

          deflecting       0.00      0.00      0.00         1
            accusing       0.90      0.90      0.90        21
            bluffing       0.65      0.83      0.73        18
            claiming       0.55      0.55      0.55        20
           defending       0.69      0.75      0.72        12
          deflecting       0.83      1.00      0.91        20
gangster ya?accusing       0.00      0.00      0.00         2
             neutral       0.85      0.69      0.76        16
          persuading       0.94      0.81      0.87        21
             probing       0.96      0.92      0.94        24

            accuracy                           0.80       155
           macro avg       0.64      0.65      0.64       155
        weighted avg       0.79      0

c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize(